# 01 — Exploratory Data Analysis
Understand the dataset before touching any model code.

In [ ]:
import sys
sys.path.insert(0, '..')   # so 'src' is importable from notebooks/

from src.config import CONFIG
from src.dataset import build_dataframes, compute_pos_weight, LABEL_MAP
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image


## 1. Load DataFrames

In [ ]:
DATA_ROOT = CONFIG.DATA_ROOT   # set this to your chest_xray/ folder path

train_df, val_df, test_df = build_dataframes(DATA_ROOT)
pos_weight = compute_pos_weight(train_df)

print(f"Train : {len(train_df):5d} images")
print(f"Val   : {len(val_df):5d} images")
print(f"Test  : {len(test_df):5d} images")
print(f"pos_weight: {pos_weight.item():.4f}")
train_df.head()


## 2. Class distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, df) in zip(axes, [('Train', train_df), ('Val', val_df), ('Test', test_df)]):
    counts = df['class_name'].value_counts()
    bars = ax.bar(counts.index, counts.values, color=['#2563eb','#dc2626'], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val}\n({100*val/len(df):.1f}%)', ha='center', va='bottom', fontsize=10)
    ax.set_title(f'{name} split', fontweight='bold')
    ax.set_ylabel('Image count')
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Class Distribution per Split', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../assets/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to assets/class_distribution.png")


## 3. Sample images — NORMAL vs PNEUMONIA

In [ ]:
def show_samples(df, class_name, n=4, title_prefix=''):
    rows = df[df['class_name'] == class_name].sample(n, random_state=42)
    fig, axes = plt.subplots(1, n, figsize=(3.5*n, 3.5))
    for ax, (_, row) in zip(axes, rows.iterrows()):
        img = Image.open(row['filepath']).convert('RGB')
        ax.imshow(img, cmap='gray')
        ax.set_title(f'{title_prefix}{class_name}', fontsize=10, color='#374151')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print("NORMAL samples:")
show_samples(train_df, 'NORMAL', n=4)
print("PNEUMONIA samples:")
show_samples(train_df, 'PNEUMONIA', n=4)


## 4. Image size distribution

In [ ]:
sizes = []
for fp in train_df['filepath'].sample(200, random_state=42):
    img = Image.open(fp)
    sizes.append(img.size)   # (width, height)

widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vals, label in [(axes[0], widths, 'Width (px)'), (axes[1], heights, 'Height (px)')]:
    ax.hist(vals, bins=20, color='#2563eb', alpha=0.8, edgecolor='white')
    ax.axvline(np.median(vals), color='#dc2626', linestyle='--', label=f'Median: {np.median(vals):.0f}')
    ax.set_xlabel(label); ax.set_ylabel('Count')
    ax.legend(); ax.spines[['top','right']].set_visible(False)

plt.suptitle('Image Size Distribution (200 random train samples)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"Width  — median: {np.median(widths):.0f}  min: {min(widths)}  max: {max(widths)}")
print(f"Height — median: {np.median(heights):.0f}  min: {min(heights)}  max: {max(heights)}")


## 5. Pixel intensity distributions

In [ ]:
def sample_pixel_stats(df, n=100):
    means, stds = [], []
    for fp in df['filepath'].sample(n, random_state=42):
        arr = np.array(Image.open(fp).convert('L'), dtype=np.float32) / 255.0
        means.append(arr.mean())
        stds.append(arr.std())
    return np.array(means), np.array(stds)

normal_means, normal_stds   = sample_pixel_stats(train_df[train_df['class_name']=='NORMAL'])
pneumo_means, pneumo_stds   = sample_pixel_stats(train_df[train_df['class_name']=='PNEUMONIA'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, n_vals, p_vals, xlabel in [
    (axes[0], normal_means, pneumo_means, 'Mean pixel intensity'),
    (axes[1], normal_stds,  pneumo_stds,  'Pixel std dev'),
]:
    ax.hist(n_vals, bins=20, alpha=0.7, color='#2563eb', label='NORMAL', edgecolor='white')
    ax.hist(p_vals, bins=20, alpha=0.7, color='#dc2626', label='PNEUMONIA', edgecolor='white')
    ax.set_xlabel(xlabel); ax.set_ylabel('Count')
    ax.legend(); ax.spines[['top','right']].set_visible(False)

plt.suptitle('Pixel Intensity Statistics by Class (100 samples each)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. EDA summary

Key observations before training:
- **Class imbalance**: ~74% PNEUMONIA vs ~26% NORMAL in train set → need `pos_weight` in loss
- **Image sizes**: variable (some are <100px, some >2000px) → Resize(256)+CenterCrop(224) is essential
- **Pixel intensity**: PNEUMONIA scans tend to have slightly higher mean intensity (consolidation/opacity)
- **Test set**: fixed 234 NORMAL / 390 PNEUMONIA — never touched during training

Ready to move to `02_baseline.ipynb` (ResNet50 baseline) → `src/train.py` (EfficientNetB3 Phase 1+2).